# LPLH2 Death Summary Prompt Test - 2026-07-05 Version

This notebook tests only the score-loss/death summarizer on fixed death histories. It does not run a 250-step game experiment.

In [ ]:
from pathlib import Path
import json, os, re, subprocess, sys, textwrap

VERSION_NAME = "lplh2_death_summary_retrieval_patch_2026-07-05"
REPO_URL = "https://github.com/ogulcaanozen/lplh2-if-agent.git"
REPO_ROOT = Path("/content/lplh2-if-agent")

if not REPO_ROOT.exists():
    print(f"Cloning {REPO_URL}...")
    subprocess.run(["git", "clone", REPO_URL, str(REPO_ROOT)], check=True)
else:
    print("Repository already exists; pulling latest main...")
    subprocess.run(["git", "-C", str(REPO_ROOT), "pull", "--ff-only"], check=True)

PROJ_ROOT = REPO_ROOT / "versions" / VERSION_NAME
assert PROJ_ROOT.exists(), f"Versioned LPLH2 root not found: {PROJ_ROOT}"
assert (PROJ_ROOT / "lplh2" / "__init__.py").exists(), "lplh2 package missing"
sys.path.insert(0, str(PROJ_ROOT))

# Default to the same local 14B path used by experiments. If you want OpenAI for this
# summarizer test, set LPLH_LLM_ES_MODEL before importing lplh2.
os.environ.setdefault("LPLH_LLM_PROVIDER", "huggingface")
os.environ.setdefault("LPLH_LLM_MODEL", "huggingface/Qwen/Qwen2.5-14B-Instruct")
os.environ.setdefault("LPLH_LLM_TEMPERATURE", "0.1")

from lplh2.llm_client import LLMClient
from lplh2 import config

print("PROJ_ROOT:", PROJ_ROOT)
print("LLM_a:", config.LLM_MODEL)
print("LLM_es:", config.LLM_ES_MODEL or "LLM_a fallback")

In [ ]:
FIXTURES = [
    {
        "name": "troll_death_after_repeated_attack",
        "reward_change": -10,
        "current_score": 15,
        "fatal_action": "attack troll with sword",
        "location": "The Troll Room",
        "ledger": """The Troll Room: visited 1 time(s) (first step 90, last step 94).
Commands already executed from this room:
- attack troll with sword x2 outcomes varied across attempts (2 different results) [last step 94]
- use sword on troll x1 INVALID: parser did not understand the command [last step 92]
Never tried from this room: any command not listed above.""",
        "history": """Step 1
Action: down
Observation: Cellar You are in a dark and damp cellar with a narrow passageway leading north.

Step 2
Action: south
Observation: East of Chasm You are on the east edge of a chasm. A narrow passage goes north.

Step 3
Action: north
Observation: Cellar You are in a dark and damp cellar with a narrow passageway leading north.

Step 4
Action: inspect sword
Observation: I don't know the word \"inspect\".

Step 5
Action: look at ramp
Observation: There's nothing special about the chute.

Step 6
Action: north
Observation: The Troll Room This is a small room with passages to the east and south and a forbidding hole leading west. A nasty-looking troll, brandishing a bloody axe, blocks all passages out of the room.

Step 7
Action: use sword on troll
Observation: I don't know the word \"use\".

Step 8
Action: attack troll with sword
Observation: A quick stroke, but the troll is on guard. The troll swings his axe, and it nicks your arm as you dodge.

Step 9
Action: attack troll with sword
Observation: You are still recovering from that last blow, so your attack is ineffective. The troll misses you.

Step 10
Action: attack troll with sword
Observation: A good stroke, but it's too slow; the troll dodges. The troll neatly removes your head. You are dead.""",
    },
    {
        "name": "troll_death_after_stab_and_swing",
        "reward_change": -10,
        "current_score": 15,
        "fatal_action": "swing sword",
        "location": "The Troll Room",
        "ledger": """The Troll Room: visited 1 time(s) (first step 110, last step 112).
Commands already executed from this room:
- use sword on troll x1 INVALID: parser did not understand the command [last step 111]
- stab troll x1 info/no useful progress: troll dodged and nicked the player's arm [last step 110]
Never tried from this room: any command not listed above.""",
        "history": """Step 1
Action: examine wooden door
Observation: The door is nailed shut.

Step 2
Action: look under trap door
Observation: There is a closed trap door under the rug.

Step 3
Action: lift trap door
Observation: The trap door opens, revealing a staircase leading down.

Step 4
Action: down
Observation: Cellar You are in a dark and damp cellar with a narrow passageway leading north.

Step 5
Action: strike trap door with sword
Observation: I've known strange people, but fighting a trap door?

Step 6
Action: use sword on trap door
Observation: I don't know the word \"use\".

Step 7
Action: north
Observation: The Troll Room This is a small room with passages to the east and south and a forbidding hole leading west. A nasty-looking troll, brandishing a bloody axe, blocks all passages out of the room.

Step 8
Action: stab troll
Observation: A good stroke, but it's too slow; the troll dodges. The troll swings his axe, and it nicks your arm as you dodge.

Step 9
Action: use sword on troll
Observation: I don't know the word \"use\".

Step 10
Action: swing sword
Observation: Whoosh! The flat of the troll's axe hits you delicately on the head, knocking you out. The troll puts you to death. You are dead.""",
    },
]

print(f"Loaded {len(FIXTURES)} death-summary fixtures.")

In [ ]:
llm = LLMClient()

def extract_json_like(text):
    match = re.search(r"\{.*\}", text or "", re.DOTALL)
    if not match:
        return None
    try:
        return json.loads(match.group(0))
    except Exception:
        return None

for fixture in FIXTURES:
    print("\n" + "=" * 90)
    print("Fixture:", fixture["name"])
    summary = llm.summarize_loss_experience(
        history=fixture["history"],
        reward_change=fixture["reward_change"],
        current_score=fixture["current_score"],
        fatal_action=fixture["fatal_action"],
        location=fixture["location"],
        command_history_block=fixture["ledger"],
    )
    print("\n--- Prompt sent ---")
    print(llm.last_summary_prompt)
    print("\n--- Parsed summary body ---")
    print(summary)
    parsed = extract_json_like(summary)
    print("\n--- JSON field check ---")
    if parsed:
        for key in ["location", "fatal_action", "proximate_cause", "confirmed_mechanics", "fatal_action_assessment", "retry_condition", "untested_idea"]:
            print(f"{key}: {parsed.get(key)}")
    else:
        print("Could not parse JSON object from summary body.")

## What to check

- `final_exchanges` should quote the last fatal turns directly.
- `confirmed_mechanics` should only include facts stated by the game or ledger.
- `retry_condition` should not say to blindly repeat the fatal command.
- `untested_idea` must stay explicitly speculative.